# Random Forest - Classificatie

## Imports

In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h2o

In [43]:
df = pd.read_csv("../data/penguins_size.csv")

In [44]:
df = df.dropna()  # droppen van de ontbrekende datapunten
df.head()

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE


In [45]:
df.sex.value_counts()

sex
MALE      168
FEMALE    165
.           1
Name: count, dtype: int64

In [46]:
df = df.loc[(df.sex != '.')]

In [47]:
df.sex.value_counts()

sex
MALE      168
FEMALE    165
Name: count, dtype: int64

In [48]:
df.columns

Index(['species', 'island', 'culmen_length_mm', 'culmen_depth_mm',
       'flipper_length_mm', 'body_mass_g', 'sex'],
      dtype='object')

## Train | Test Split

In [51]:
# Define features (X) and target (y)
X = df.drop("species", axis=1)
y = df["species"]

In [54]:
from sklearn.model_selection import train_test_split

In [55]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [56]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# Apply one-hot encoding and standard scaling 
column_transformer = ColumnTransformer(
	transformers=[
		('onehot', OneHotEncoder(drop='first'), ['island', 'sex']),
		('scaler', StandardScaler(), ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g'])
	],
	remainder='passthrough'
)

X_train = column_transformer.fit_transform(X_train)
X_test = column_transformer.transform(X_test)

In [29]:
h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; OpenJDK Client VM (Temurin)(build 25.432-b06, mixed mode))
  Starting server from C:\Users\Cursist\DataScienceCourse\DS_Projects\venv_ml\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\Cursist\AppData\Local\Temp\tmpz1d80yy6
  JVM stdout: C:\Users\Cursist\AppData\Local\Temp\tmpz1d80yy6\h2o_Cursist_started_from_python.out
  JVM stderr: C:\Users\Cursist\AppData\Local\Temp\tmpz1d80yy6\h2o_Cursist_started_from_python.err


c:\Users\Cursist\DataScienceCourse\DS_Projects\venv_ml\Lib\site-packages\h2o\backend\server.py:395: UserWarning:   You have a 32-bit version of Java. H2O works best with 64-bit Java.
  Please download the latest 64-bit Java SE JDK from Oracle.

  warn("  You have a 32-bit version of Java. H2O works best with 64-bit Java.\n"


  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,07 secs
H2O_cluster_timezone:,Europe/Paris
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,4 days
H2O_cluster_name:,H2O_from_python_Cursist_2neusn
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,247.5 Mb
H2O_cluster_total_cores:,0
H2O_cluster_allowed_cores:,0
H2O_cluster_status:,"locked, healthy"


In [59]:
# Convert numpy arrays back to pandas DataFrame
X_train_df = pd.DataFrame(X_train, columns=column_transformer.get_feature_names_out())
X_test_df = pd.DataFrame(X_test, columns=column_transformer.get_feature_names_out())

# Split the H2O Frame into training and testing sets
h2o_train = h2o.H2OFrame(pd.concat([X_train_df, y_train.reset_index(drop=True)], axis=1))
h2o_test = h2o.H2OFrame(pd.concat([X_test_df, y_test.reset_index(drop=True)], axis=1))

# Ensure the target column is treated as categorical
h2o_train["species"] = h2o_train["species"].asfactor()
h2o_test["species"] = h2o_test["species"].asfactor()

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [61]:
from h2o.automl import H2OAutoML
from sklearn.metrics import precision_score, recall_score

# Run H2O AutoML
aml = H2OAutoML(max_runtime_secs=300, seed=42)  # Set max runtime to 5 minutes (300 seconds)
aml.train(x=list(X_train_df.columns), y="species", training_frame=h2o_train)

AutoML progress: |
15:50:47.316: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2ODeepLearningEstimator : Deep Learning
Model Key: DeepLearning_grid_3_AutoML_1_20260316_155047_model_1


Status of Neuron Layers: predicting species, 3-class classification, multinomial distribution, CrossEntropy loss, 1.063 weights/biases, 18,2 KB, 2.353 training samples, mini-batch size 1
    layer    units    type              dropout    l1    l2    mean_rate    rate_rms    momentum    mean_weight    weight_rms    mean_bias    bias_rms
--  -------  -------  ----------------  ---------  ----  ----  -----------  ----------  ----------  -------------  ------------  -----------  ----------
    1        7        Input             15
    2        20       RectifierDropout  0          0     0     0.00145673   0.00103057  0           -0.0496752     0.267331      0.513027     0.0298649
    3        20       RectifierDropout  0          0     0     0.00166667   0.00150355  0           0.00419719     0.220959      1.00464      0.0440107
    4        20       RectifierDropout  0          0     0     0.05389      0.221424    0           0.00413934     0.214764      0.991993     0.0216493
    5        3        Softmax                      0     0     0.0562545    0.216461    0           -0.129158      1.20077       6.17654e-05  0.0241246

ModelMetricsMultinomial: deeplearning
** Reported on train data. **

MSE: 0.004390616630030215
RMSE: 0.06626172824512061
LogLoss: 0.019686570011167807
Mean Per-Class Error: 0.006060606060606061
AUC table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).
AUCPR table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).

Confusion Matrix: Row labels: Actual class; Column labels: Predicted class
Adelie    Chinstrap    Gentoo    Error      Rate
--------  -----------  --------  ---------  -------
115       0            0         0          0 / 115
1         54           0         0.0181818  1 / 55
0         0            96        0          0 / 96
116       54           96        0.0037594  1 / 266

Top-3 Hit Ratios: 
k    hit_ratio
---  -----------
1    0.996241
2    1
3    1

ModelMetricsMultinomial: deeplearning
** Reported on cross-validation data. **

MSE: 0.0027105836928086715
RMSE: 0.05206326625182742
LogLoss: 0.013503785486277162
Mean Per-Class Error: 0.0
AUC table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).
AUCPR table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).

Confusion Matrix: Row labels: Actual class; Column labels: Predicted class
Adelie    Chinstrap    Gentoo    Error    Rate
--------  -----------  --------  -------  -------
115       0            0         0        0 / 115
0         55           0         0        0 / 55
0         0            96        0        0 / 96
115       55           96        0        0 / 266

Top-3 Hit Ratios: 
k    hit_ratio
---  -----------
1    1
2    1
3    1

Cross-Validation Metrics Summary: 
                         mean        sd          cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
-----------------------  ----------  ----------  ------------  ------------  ------------  ------------  ------------
accuracy                 0.988749    0.0168376   0.981482      1             1             1             0.962264
aic                      nan         0           nan           nan           nan           nan           nan
auc                      nan         0           nan           nan           nan           nan           nan
err                      0.0112509   0.0168376   0.0185185     0             0             0             0.0377359
err_count                0.6         0.894427    1

In [64]:
# Who are the leaders?
lb = aml.leaderboard
print(lb.head(rows=5))

model_id                                                mean_per_class_error    logloss       rmse         mse
DeepLearning_grid_3_AutoML_1_20260316_155047_model_1              0           0.0135038  0.0520633  0.00271058
DeepLearning_grid_1_AutoML_1_20260316_155047_model_2              0.0057971   0.0109389  0.062279   0.00387868
GLM_1_AutoML_1_20260316_155047                                    0.0057971   0.0179389  0.074193   0.0055046
DeepLearning_grid_2_AutoML_1_20260316_155047_model_2              0.00606061  0.0190729  0.0709607  0.00503543
GBM_grid_1_AutoML_1_20260316_155047_model_58                      0.00606061  0.02537    0.0755707  0.00571093
[5 rows x 5 columns]

